# Codex research backend

Install AutoHedge from this checkout (`pip install -e .`). The offline preview below does not launch Codex, access API providers, fetch market data, or submit orders.

The optional live cell requires the official Codex CLI (`npm install -g @openai/codex`) and `codex login`. It uses six model requests against your existing Codex allowance. Use supplied, dated evidence for real research; this example uses fictional data.

Codex mode has a fixed research pipeline: director thesis → sentiment → quant → risk → order proposal → director synthesis. It does not implement native Swarms handoffs or Exa search. The normal `litellm` backend remains available.


In [ ]:
import os
from tempfile import TemporaryDirectory
from unittest.mock import patch

os.environ["SWARMS_TELEMETRY_ON"] = "false"

from autohedge import AutoHedge
from autohedge.codex_research import CodexResearchDirector

task = (
    "Use only this fictional dataset: SYNTH closes [100, 101, 99, 102], "
    "daily volume [1000, 1200, 900, 1100]. Identify missing evidence. "
    "No news or fundamentals are supplied. Do not recommend a live trade."
)

with TemporaryDirectory() as directory:
    with patch(
        "autohedge.codex_research.shutil.which",
        return_value="offline-codex",
    ):
        with patch.object(
            CodexResearchDirector,
            "_generate",
            return_value="Offline sample report",
        ) as generate:
            system = AutoHedge(
                llm_backend="codex",
                codex_config={"model": None, "timeout": 180},
                output_type="str",
                output_dir=directory,
            )
            print(system.run(task))
            assert generate.call_count == 6


In [ ]:
# Change to True to use your authenticated Codex CLI and account quota.
RUN_LIVE = False

if RUN_LIVE:
    system = AutoHedge(
        llm_backend="codex",
        codex_config={"model": None, "timeout": 180},
        output_type="str",
    )
    print(system.run(task))
